## Checklist / Persistance setup

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, re, logging
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2,3,4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from rich.console import Console, Group
from rich.theme import Theme
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich.box import SIMPLE_HEAVY
from rich.markup import escape
from src.utils import logging_utils, env_utils

logging_utils.configure(level=logging.WARNING)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)
RES = env_utils.DEFAULT_RESULTS_DIR

In [2]:
# ── model ──
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"

mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
)
print(mt.name)

2026-08-19 17:32:02 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct


In [14]:
import torch
from src.utils import env_utils


def mask_bin(heads):
    m = HeadMask(mt.n_layer, mt.model.config.num_attention_heads)
    with torch.no_grad():
        m.values.zero_()
        for l, h in heads:
            m.values[l, h] = 1.0
    return m


d = torch.load(
    os.path.join(
        env_utils.DEFAULT_RESULTS_DIR,
        "notebook_caches/15_order_dependency/aff_sep__d5fbad08ba.pt",
    ),
    map_location="cpu",
    weights_only=False,
)["value"]
boundary_heads = sorted(lh for lh, c in d.items() if c < 0.7)
DEC_MASK = mask_bin(boundary_heads)
print(f'Active heads = {DEC_MASK.n_active()}')

MEAN_MOVEON, MEAN_STAY = torch.load(
    os.path.join(
        env_utils.DEFAULT_RESULTS_DIR,
        "notebook_caches/15_order_dependency/means__e542a9c825.pt",
    ),
    map_location=mt.device,
    weights_only=False,
)["value"]
print(MEAN_MOVEON.shape, MEAN_MOVEON.norm())

Active heads = 140
torch.Size([64, 5120]) tensor(210.7545, device='cuda:0')


In [15]:
# ── goal heads: load the trained head mask (same recipe as clock/03_oid) ──
from src.goals.head_loc_attn import HeadMask

MASK_PATH = os.path.join(RES, f"localization/{MODEL_NAME.split('/')[-1]}/lam20/head_mask.pt")
ckpt = torch.load(MASK_PATH, map_location="cpu")
LAYERS = ckpt["layers"]
n_heads = ckpt["mask_values"].shape[1]
mask = HeadMask(len(LAYERS), n_heads)
with torch.no_grad():
    mask.values.copy_(ckpt["mask_values"])
    if mt.name.split("/")[-1] == "Qwen2.5-Coder-32B-Instruct":
        mask.values[0:25] = 0     # keep the middle band only
        mask.values[56:] = 0

THRESHOLD = 0.9
goalscope_heads = sorted(mask.selected_heads(LAYERS, threshold=THRESHOLD))
print(f"loaded {MASK_PATH}")
print(f"layers {LAYERS[0]}–{LAYERS[-1]} · active heads (mask>{THRESHOLD}): {len(goalscope_heads)}")

loaded /disk/u/arnab/Codes/Projects/goals/results/localization/Qwen2.5-Coder-32B-Instruct/lam20/head_mask.pt
layers 0–63 · active heads (mask>0.9): 122


In [47]:
from src.tokens import prepare_input

def exact_prompt(prompt, tail_tokens=None, patch_pos=[-2]):
    """The prompt EXACTLY as the model sees it, with PATCH_POS marked.

    Special tokens, the augmented `    \n` padding and the trailing spaces are all shown — that
    padding is precisely what PATCH_POS indexes into, so a cleaned-up rendering hides the one thing
    worth checking. Newlines print as a literal `\n` AND wrap.

    `escape()` is applied ONCE to the whole joined string, never per token: a lone `[` token is not
    escaped on its own, so per-token escaping lets `[` and the following token re-form a rich tag —
    which silently ate `list[str]` and `[w for w in words ...]` out of the printed prompt.

    `tail_tokens=N` shows only the last N tokens (the chat preamble is identical across prompts;
    the tail is where 3a and 3b differ). Returns rich markup.
    """
    ids = prepare_input(prompts=prompt, tokenizer=mt).input_ids[0].tolist()
    piv = [len(ids) + p for p in patch_pos]             # PATCH_POS is negative -> absolute index
    lo = 0 if tail_tokens is None else max(0, len(ids) - tail_tokens)
    OPEN, CLOSE = "\x00<", "\x00>"                    # sentinels survive escape(); tags do not
    parts = []
    for i in range(lo, len(ids)):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        parts.append(OPEN + s + CLOSE if i in piv else s)
    txt = escape("".join(parts))
    txt = txt.replace(OPEN, "[black on yellow]").replace(CLOSE, "[/black on yellow]")
    head = f"[grey42]… {lo} earlier tokens elided …[/grey42]\n" if lo else ""
    return (head + txt +
            f"\n[grey58](PATCH_POS={patch_pos} -> absolute token {piv} of {len(ids)}; "
            f"highlighted above)[/grey58]")

def pivot_token(prompt, patch_pos=[-2]):
    ids = prepare_input(prompts=prompt, tokenizer=mt).input_ids[0].tolist()
    return [mt.tokenizer.decode([ids[len(ids) + p]]) for p in patch_pos]

In [90]:
from src.functional import generate_with_patch
from src.goals.goalscope import headpatched_generate

# question = """You need to load 3 .txt files and concatanate their contents into a single string. Here are the file names (to be concatenated in the order given): summary.txt, info.txt, bash.txt. Write a Python function to do so."""
# prefill = """<code>
# def concatenate_files():
#     f1 = open("info.txt", "r").read()
#     f2 = open("bash.txt", "r")
# """

# question = """Write a C++ function that takes a list of integer numbers as a list and returns a stack. Accessing and popping elements from the stack should return the elements in sorted order (from smallest to largest). The original list is not sorted."""
# prefill = """<code>
# #include <stack>
# #include <vector>
# #include <algorithm>

# std::stack<int> function(const std::vector<int>& nums) {
#     std::vector<int> sorted = nums;
#     std::stack<int> stk;
#     stk.push(nums[0]);
# """

# question = """Write a Python program that implements a function `check_conditions` which, given a list of strings `words`, computes the following:

# 1. check if the words in `words` are in alphabetical order.
# 2. the length of each word in `words`, in descending order.
# 3. the longest word in `words` starts with a vowel.

# For each item, in the order listed, assign the result to its own local variable, using single lowercase letters in order — `a` for the first item, `b` for the second, and so on. After all items have been computed, use a SINGLE print statement that, using those variables, reports each result. `print(f"a: {a}, b: {b}", ...)`
# Compute every value inside `check_conditions` (one local variable per item); do not define any helper functions.
# Do not include any comments in your code."""
# prefill = """<code>
# def check_conditions(words):
#     a = sorted(words)
#     a = a == words
#     b = lambda x: -len(x)
#     c = max(words, key=len).lower()[0] in 'aeiou'
#     print("""
# PATCH_POS = [-22]


# question = """Write a Python program that implements a function `check_conditions` which, given a list of strings `words`, computes the following:

# 1. check if the words in `words` are in alphabetical order.
# 2. each word capitalized and joined into a single string, separated by spaces.
# 3. the longest word in `words` starts with a vowel.

# For each item, in the order listed, assign the result to its own local variable, using single lowercase letters in order — `a` for the first item, `b` for the second, and so on. After all items have been computed, use a SINGLE print statement that, using those variables, reports each result. `print(f"a: {a}, b: {b}", ...)`
# Compute every value inside `check_conditions` (one local variable per item); do not define any helper functions.
# Do not include any comments in your code."""
# prefill = """<code>
# def check_conditions(words):
#     a = sorted(words)
#     a = a == words
#     b = [word.capitalize() for word in words]
#     c = max(words, key=len).lower()[0] in 'aeiou'
#     print(f"a: {a}, b:"""
# PATCH_POS = [-29, -30, -31]


question = """Write a Python program that implements a function `check_conditions` which, given a list of strings `words`, computes the following:

1. the number of words that contain the letter 'a' (case-insensitive).
2. the first 2 characters of each word in `words`, joined into a single string, separated by commas.
3. the longest word in `words` starts with a vowel.

For each item, in the order listed, assign the result to its own local variable, using single lowercase letters in order — `a` for the first item, `b` for the second, and so on. After all items have been computed, use a SINGLE print statement that, using those variables, reports each result.
Compute every value inside `check_conditions` (one local variable per item); do not define any helper functions.
Do not include any comments in your code."""
prefill = """<code>
def check_conditions(words):
    a = [word for word in words if 'a' in word.lower()]
    b = [word[:2] for word in words]
    c = max(words, key=len).lower()[0] in 'aeiou'
    print(f"a:"""
PATCH_POS = [-24, -37]


messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks.",
        # "content": "You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. No comments or explanations, just the code.",
    },
    {"role": "user", "content": question},
    {"role": "assistant", "content": prefill},
]
demo_prompt = mt.tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=False,
    continue_final_message=True,
    tokenize=False,
    chat_template_kwargs={"enable_thinking": False},
)


hdr = (
    f"pivot [yellow]{pivot_token(demo_prompt, patch_pos=PATCH_POS)!r}[/yellow] at PATCH_POS={PATCH_POS}   ·   "
    f"[salmon1]{DEC_MASK.n_active()}[/salmon1] heads   ·   "
    f"replace, n_steer_tokens=1"
)
rich_print(
    Panel(
        Group(
            Text.from_markup(hdr),
            Text(""),
            Text("exact prompt as the model sees it:", style="bold"),
            Text.from_markup(exact_prompt(demo_prompt, patch_pos=PATCH_POS)),
        ),
        title="[bold]prompt[/bold]",
        title_align="left",
        border_style="grey42",
        padding=(1, 1),
    )
)

╭─ prompt ────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ pivot [']\n', '()]\n'] at PATCH_POS=[-24, -37]   ·   140 heads   ·   replace, n_steer_tokens=1                  │
│                                                                                                                 │
│ exact prompt as the model sees it:                                                                              │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code>               │
│ blocks.<|im_end|>\n                                                                                             │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `check_conditions` which, given a list of strings `words`,    │
│ computes the following:\n                                                                                       │
│ \n                                                                                                              │
│ 1. the number of words that contain the letter 'a' (case-insensitive).\n                                        │
│ 2. the first 2 characters of each word in `words`, joined into a single string, separated by commas.\n          │
│ 3. the longest word in `words` starts with a vowel.\n                                                           │
│ \n                                                                                                              │
│ For each item, in the order listed, assign the result to its own local variable, using single lowercase letters │
│ in order — `a` for the first item, `b` for the second, and so on. After all items have been computed, use a     │
│ SINGLE print statement that, using those variables, reports each result.\n                                      │
│ Compute every value inside `check_conditions` (one local variable per item); do not define any helper           │
│ functions.\n                                                                                                    │
│ Do not include any comments in your code.<|im_end|>\n                                                           │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def check_conditions(words):\n                                                                                  │
│     a = [word for word in words if 'a' in word.lower()]\n                                                       │
│     b = [word[:2] for word in words]\n                                                                          │
│     c = max(words, key=len).lower()[0] in 'aeiou'\n                                                             │
│     print(f"a:                                                                                                  │
│ (PATCH_POS=[-24, -37] -> absolute token [250, 237] of 274; highlighted above)                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [91]:
clean_gen = generate_with_patch(
    mt = mt,
    inputs = demo_prompt,
    max_new_tokens = 256,
    remove_prefix=True,
    n_gen_per_prompt=1,
    do_sample=False,
)[0]
rich_print(f"[sea_green1]{escape(clean_gen)}[/sea_green1]")

ALPHAS = [1.0, 2.0, 4.0, 6.0]
for alpha in ALPHAS:
    patched_gen = headpatched_generate(
        mt=mt,
        dest_inputs=dict(prepare_input(prompts=demo_prompt, tokenizer=mt)),
        src_o_proj_inps = MEAN_MOVEON,
        mask=DEC_MASK,
        max_new_tokens=256,
        n_steer_tokens=1,
        patch_input_positions=PATCH_POS,
        alpha=alpha,
    )
    rich_print(f"[salmon1][bold]{alpha=}[/bold]\n{escape(patched_gen)}[/salmon1]")

 {len(a)}, b: {','.join(b)}, c: {c}")
</code>

alpha=1.0
 {len(a)}, b: {','.join(b)}, c: {c}")
</code>

alpha=2.0
 {a}, b: {b}, c: {c}")
</code>

alpha=4.0
 {a}, b: {','.join(b)}, c: {c}")
</code>

alpha=6.0
 {a}, b: {','.join(b)}, c: {c}")
</code>

In [38]:
# ── custom goalscope destination prompts (question + assistant prefill) ──────
def build_goalscope_prompt(mt, question, prefill=""):
    # chat-template prompt whose assistant turn is pre-seeded with `prefill` for the model to
    # continue -- same structure as goalscope._build_goalscope_prompt, but with our own wording.
    messages = [
        {"role": "system", "content": "You are a helpful assistant proficient in programming."},
        {"role": "user", "content": question},
        {"role": "assistant", "content": prefill},
    ]
    try:
        return mt.tokenizer.apply_chat_template(
            messages, add_generation_prompt=False, continue_final_message=True,
            tokenize=False, chat_template_kwargs={"enable_thinking": False})
    except TypeError:
        return mt.tokenizer.apply_chat_template(
            messages, add_generation_prompt=False, continue_final_message=True, tokenize=False)

## Run — one prompt, one phrasing

Set the two knobs and run. The cell generates the code the model actually writes for
`DEMO_PROMPT_KEY`, then GoalScopes the whole `<code>` segment through `GOALSCOPE_KEY`'s phrasing.
Read the resulting HTML by eye — hover a token to see its verbalization. The tokens to watch are
the start of the consumer line (`c = …`): does the read-out there name `b` / the producer's
construction?

In [39]:
GOALSCOPE_VARIANTS = {
    "default": dict(
        question="What are you doing at this line of the code. Keep your answer as brief as possible.",
        prefill="",
    ),
    "depend_on": dict(
        question="At this point in the code, which earlier computed value does this line depend on? Answer in a few words.",
        prefill="Here I depend on",
    ),
    "waiting_for": dict(
        question="At this point in the code, what earlier result are you waiting for (if any) before you can continue? If not waiting for anything, answer 'nothing'.",
        prefill="I am waiting for",
    ),
    "reuse": dict(
        question="At this point in the code, which earlier variable's result are you about to reuse? Answer in a few words.",
        prefill="I am about to reuse",
    ),
}

GOALSCOPE_KEY = "default"
gv = GOALSCOPE_VARIANTS[GOALSCOPE_KEY]
gsp = build_goalscope_prompt(mt, gv["question"], gv["prefill"])
print(gsp)

<|im_start|>system
You are a helpful assistant proficient in programming.<|im_end|>
<|im_start|>user
What are you doing at this line of the code. Keep your answer as brief as possible.<|im_end|>
<|im_start|>assistant



In [40]:
from src.functional import generate_with_patch
from src.goals.goalscope import apply_goalscope

# ── knobs ───────────────────────────────────────────────────────────────────
STRATEGY, ALPHA = "additive_norm", 0.1      # goalscope injection ("additive" | "additive_norm" | "replace")
STEER_LAYERS    = (35, 46)                  # additive strategies only
GEN_MAXNEW      = 512                       # tokens to write the full function body
SCOPE_MAXNEW    = 128                       # tokens per verbalization
SCOPE_BATCH     = 64                        # batch size for verbalization

# 2. verbalize every code token through the chosen phrasing
print(f"\nphrasing [{GOALSCOPE_KEY}]  prefill={gv['prefill']!r}\n")
res = apply_goalscope(
    prompt = demo_prompt + gen, mt=mt, selected_heads=goalscope_heads,
    strategy=STRATEGY, alpha=ALPHA, steer_layer_range=STEER_LAYERS,
    batch_size=SCOPE_BATCH, max_new_tokens=SCOPE_MAXNEW,
    goalscope_prompt=gsp,
    include_noncode=True,
    title=f"{GOALSCOPE_KEY} · (prefill: {gv['prefill']!r})",
)


phrasing [default]  prefill=''



GoalScope[additive_norm]:   0%|          | 0/2 [00:00<?, ?it/s]